In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
!pip install scikit-learn nltk
!pip install openai
!pip install -q git+https://github.com/huggingface/peft.git transformers bitsandbytes datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 12.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 12.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 10.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [17]:
%run fine_tuned_blip_inference.py
%run few_shot_instruction_generation.py

In [4]:
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
scoring_model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

def get_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        outputs = scoring_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.numpy()

def calculate_cosine_sim(text1,text2):
  text1_embedding = get_embedding(text1)
  text2_embedding = get_embedding(text2)

  # Calculate cosine similarity
  cosine_sim = cosine_similarity(text1_embedding, text2_embedding)
  return cosine_sim[0][0]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

In [13]:
import csv
from prettytable import PrettyTable

testset = '/content/caption_testset.csv'
image_folder = "/content/drive/MyDrive/LLM final project/testset_imagecap"

table = PrettyTable()
table.field_names = ["Filename", "True Caption", "Generated Caption", "Similarity Score"]


cosines = []
with open(testset, mode='r', newline='') as file:
    reader = csv.reader(file)

    next(reader)

    for i,row in  enumerate(reader):
        image_path = f"{image_folder}/{row[0]}"
        image = Image.open(image_path)
        true_caption = f"grasping location the {row[2]}"
        caption = generate_caption(image,fine_tuned_model)
        score = calculate_cosine_sim(true_caption,caption)
        cosines.append(score)
        table.add_row([row[0], true_caption , caption,score ])



In [14]:
print(table)
print("Average Cosine Similarity Score: ", sum(cosines)/len(cosines))


+----------------+-----------------------------------------+---------------------------------------+------------------+
|    Filename    |               True Caption              |           Generated Caption           | Similarity Score |
+----------------+-----------------------------------------+---------------------------------------+------------------+
|  bottle1.jpeg  |   grasping location the bottle's neck   |   grasping location a bottle's neck   |    0.99040616    |
|  bottle4.jpeg  |   grasping location the bottle's neck   |   grasping location a bottle's neck   |    0.99040616    |
|  bottle5.jpeg  |   grasping location the bottle's body   |   grasping location a bottle's body   |    0.99080884    |
|  bottle6.jpeg  |   grasping location the bottle's body   |   grasping location a bottle's body   |    0.99080884    |
|  bottle6.jpeg  |   grasping location the bottle's body   |   grasping location a bottle's body   |    0.99080884    |
|  bottle7.jpeg  |   grasping location t

In [18]:
testset = '/content/expanded_prompts_and_instructions.csv'

table = PrettyTable()
table.field_names = ["Prompt", "True Instructions", "Generated Instructions", "Similarity Score"]


cosines = []
with open(testset, mode='r', newline='') as file:
    reader = csv.reader(file)

    next(reader)

    for i,row in  enumerate(reader):
        true_instruction  = f"task: {row[2]}\nobject: {row[1]}\ngrasping location: {row[3]}"

        generated_instruction = generate_instruction(row[0],f'image of a {row[1]}')

        score = calculate_cosine_sim(true_instruction,generated_instruction)
        cosines.append(score)
        table.add_row([row[0], true_instruction , generated_instruction,score ])

In [19]:
print(table)
print("Average Cosine Similarity Score: ", sum(cosines)/len(cosines))


+------------------------------------+---------------------------------------------------+-----------------------------------------------------------------+------------------+
|               Prompt               |                 True Instructions                 |                      Generated Instructions                     | Similarity Score |
+------------------------------------+---------------------------------------------------+-----------------------------------------------------------------+------------------+
|         Hand me the remote         |         task: Bring the remote to the user        |                task: pass the remote to the user                |    0.97470474    |
|                                    |                   object: remote                  |                          object: remote                         |                  |
|                                    |          grasping location: Remote’s body         |                 grasping loca